# Seminar 2. Custom PyTorch Operators

# Building Models in PyTorch Through Composition

PyTorch models are built using **composition**.  
Instead of defining one large monolithic network, we construct models by combining smaller, reusable modules.

Each module can contain other modules, which allows us to build hierarchical and well-structured architectures.

---

## Composition

Composition means:

- A model is built from smaller blocks.
- Each block can contain multiple layers.
- Blocks can be reused in larger architectures.
- Complex models are created by stacking simpler components.

This keeps code:

- Modular  
- Reusable  
- Readable  
- Easy to extend  


## Key Ideas

- Inherit from `nn.Module`
- Define layers inside `__init__`
- Define computation in `forward()`
- Create reusable blocks
- Build larger models by combining blocks

---

## Example: Model Built from Two Blocks

Below is a simple example where:

- We define a reusable blocks: `LinearReLUBlock` and `LinearTanhBlock`
- The final model is composed of two such blocks


In [1]:
import torch
import torch.nn as nn


class LinearReLUBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.ReLU()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class LinearTanhBlock(nn.Module):
    def __init__(self, in_features: int, out_features: int):
        super().__init__()

        self.linear = nn.Linear(in_features, out_features)
        self.activation = nn.Tanh()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = self.activation(x)
        return x


class CombinedModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.block1 = LinearReLUBlock(4, 8)
        self.block2 = LinearTanhBlock(8, 8)
        self.output = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.block1(x)
        x = self.block2(x)
        x = self.output(x)
        return x


model = CombinedModel()
print(model)

CombinedModel(
  (block1): LinearReLUBlock(
    (linear): Linear(in_features=4, out_features=8, bias=True)
    (activation): ReLU()
  )
  (block2): LinearTanhBlock(
    (linear): Linear(in_features=8, out_features=8, bias=True)
    (activation): Tanh()
  )
  (output): Linear(in_features=8, out_features=2, bias=True)
)


# What `nn.Module` Enables

When we inherit from `nn.Module`, we automatically gain powerful functionality that works **recursively across all submodules**.

## What `nn.Module` Gives Us



### Parameter Registration

All layers assigned as attributes (e.g. `self.linear = nn.Linear(...)`) are:

- Automatically registered
- Collected in `model.parameters()`
- Included in `model.state_dict()`

This works **recursively** for all sub-blocks.

In [2]:
print("Registered parameters:")
for name, param in model.named_parameters():
    print(name, param.shape)


Registered parameters:
block1.linear.weight torch.Size([8, 4])
block1.linear.bias torch.Size([8])
block2.linear.weight torch.Size([8, 8])
block2.linear.bias torch.Size([8])
output.weight torch.Size([2, 8])
output.bias torch.Size([2])


### Automatic Gradient Tracking

During the forward pass:

- PyTorch dynamically builds a computation graph
- Calling `loss.backward()` computes gradients
- Gradients are stored in each parameter’s `.grad`

No manual graph management is required.

In [3]:
x = torch.randn(5, 4)
target = torch.randn(5, 2)

criterion = nn.MSELoss()
output = model(x)
loss = criterion(output, target)

loss.backward()

print("\nGradient computed for output layer:",
      model.output.weight.grad is not None)
model.output.weight.grad


Gradient computed for output layer: True


tensor([[-0.1943, -0.1282,  0.2531,  0.1220, -0.1597, -0.0948, -0.0906, -0.1851],
        [ 0.1488,  0.1000, -0.1775, -0.1313,  0.1392,  0.1140,  0.0599,  0.1432]])

### Device and Type Transfer (`.to()`)

Calling:

    model.to(device)

or

    model.to(dtype)

moves all:

- Parameters
- Buffers
- Submodules

to CPU/GPU/dtype automatically.

In [4]:
import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

dtype = torch.float32

model = model.to(device=device, dtype=dtype)

x: torch.Tensor = torch.randn(5, 4, device=device, dtype=dtype)

print("Model device:", next(model.parameters()).device)
print("Model dtype:", next(model.parameters()).dtype)

Model device: cuda:0
Model dtype: torch.float32


### Saving & Loading (`state_dict()`)

- `model.state_dict()` returns all parameters recursively
- `model.load_state_dict(...)` restores them

This works across the full module tree.

In [5]:
state_dict = model.state_dict()
torch.save(state_dict, "combined_model.pt")

# `train()` vs `eval()` Mode in PyTorch

PyTorch modules have two main modes: **training mode** and **evaluation mode**.  
Switching between them affects layers that behave differently during training and inference.

---

## `model.train()`

- Sets the model to **training mode**.
- Used when training the model with gradient updates.
- Affects certain layers, such as:

| Layer Type        | Behavior in `train()` Mode                  |
|------------------|--------------------------------------------|
| `Dropout`         | Randomly zeroes some activations           |
| `BatchNorm`       | Updates running statistics (mean/variance) |

- Gradients are computed as usual.

---

## `model.eval()`

- Sets the model to **evaluation (inference) mode**.
- Used when evaluating or deploying the model.
- Affects certain layers:

| Layer Type        | Behavior in `eval()` Mode                   |
|------------------|--------------------------------------------|
| `Dropout`         | Passes all activations through unchanged  |
| `BatchNorm`       | Uses stored running mean/variance         |

- No layers update internal statistics.
- Gradients are usually not required (often used with `torch.no_grad()`).

---

## Key Points

- Always use `model.train()` during training.
- Always use `model.eval()` during evaluation or testing.
- Forgetting to switch can lead to inconsistent results, especially with `Dropout` or `BatchNorm`.



In [6]:
import torch
import torch.nn as nn

# Simple model with Dropout and BatchNorm
class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc1 = nn.Linear(4, 8)
        self.bn = nn.BatchNorm1d(8)
        self.dropout = nn.Dropout(p=0.5)
        self.fc2 = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.bn(x)
        x = torch.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x


model = SimpleModel()
x = torch.randn(5, 4)

# Training mode
model.train()
output_train = model(x)
print("Training mode output:\n", output_train)

# Evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)
print("Evaluation mode output:\n", output_eval)


Training mode output:
 tensor([[ 0.6965,  0.1485],
        [-0.2665,  0.1725],
        [ 0.9418,  0.4853],
        [-0.5367, -0.0944],
        [-0.1579,  0.2844]], grad_fn=<AddmmBackward0>)
Evaluation mode output:
 tensor([[-0.3293,  0.7248],
        [-0.5042,  0.5324],
        [ 0.1359,  0.6565],
        [-0.1586,  0.2416],
        [-0.1826,  0.2823]])


# `torch.no_grad()` and `torch.inference_mode()` in PyTorch

When performing inference (evaluating a model without updating parameters), PyTorch provides context managers to **disable gradient tracking**. This saves memory and speeds up computation.

---

## `torch.no_grad()`

- Disables gradient tracking.
- Useful during evaluation or inference.
- Gradients are **not computed**, but autograd still tracks operations for some internal purposes.
- Can be used as a **context manager** or a **function decorator**.

---

## `torch.inference_mode()`

- Introduced in PyTorch 1.9.
- Similar to `no_grad()`, but **more efficient**.
- Completely disables autograd and reduces memory usage.
- Recommended for pure inference pipelines.



In [8]:
import torch
import torch.nn as nn

class SimpleModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.fc = nn.Linear(4, 2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.no_grad() as method decorator
    # -----------------------------
    @torch.no_grad()
    def forward_no_grad(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)

    # -----------------------------
    # Using torch.inference_mode() as method decorator
    # -----------------------------
    @torch.inference_mode()
    def forward_inference(self, x: torch.Tensor) -> torch.Tensor:
        return self.fc(x)


model = SimpleModel()
x = torch.randn(5, 4)

# -----------------------------
# Call decorated methods
# -----------------------------
output_no_grad_method = model.forward_no_grad(x)
output_infer_method = model.forward_inference(x)

print("Output no_grad method:\n", output_no_grad_method)
print("Output inference_mode method:\n", output_infer_method)

# -----------------------------
# Using context managers
# -----------------------------

with torch.no_grad():
    output_no_grad_cm = model(x)

with torch.inference_mode():
    output_infer_cm = model(x)

print("Output no_grad context manager:\n", output_no_grad_cm)
print("Output inference_mode context manager:\n", output_infer_cm)

Output no_grad method:
 tensor([[ 0.2518, -0.6974],
        [ 0.2072, -0.0785],
        [-0.2209,  1.2734],
        [-0.2958,  0.7715],
        [ 0.1042,  0.7625]])
Output inference_mode method:
 tensor([[ 0.2518, -0.6974],
        [ 0.2072, -0.0785],
        [-0.2209,  1.2734],
        [-0.2958,  0.7715],
        [ 0.1042,  0.7625]])
Output no_grad context manager:
 tensor([[ 0.2518, -0.6974],
        [ 0.2072, -0.0785],
        [-0.2209,  1.2734],
        [-0.2958,  0.7715],
        [ 0.1042,  0.7625]])
Output inference_mode context manager:
 tensor([[ 0.2518, -0.6974],
        [ 0.2072, -0.0785],
        [-0.2209,  1.2734],
        [-0.2958,  0.7715],
        [ 0.1042,  0.7625]])


# Disabling Gradients with `requires_grad_(False)`

PyTorch provides a convenient method `requires_grad_()` that can **enable or disable gradients in-place** for all parameters of a model or a tensor.

Using:

```python
param.requires_grad_(False)
```

- Sets `requires_grad=False` **in-place** for that parameter.
- This is useful for freezing models during inference or transfer learning.
- Can be applied to an entire model recursively by iterating over its parameters.


In [9]:
model = SimpleModel()

# Disable gradient computation for all parameters using requires_grad_()
for param in model.parameters():
    param.requires_grad_(False)

# Verify
for name, param in model.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

# Forward pass still works
x = torch.randn(5, 4)
output = model(x)
print("Output shape:", output.shape)

fc.weight: requires_grad=False
fc.bias: requires_grad=False
Output shape: torch.Size([5, 2])


# Redefining `train()` and `eval()` in `nn.Module`

PyTorch’s `nn.Module` provides built-in `train(mode: bool = True)` and `eval()` methods to switch between **training** and **evaluation** modes.  

Sometimes, when creating **custom modules or blocks**, you might want to **override these methods** to perform extra actions whenever the mode changes.

---

## Why Override?

- Apply mode-specific logic to sub-blocks or attributes that are not standard layers
- Log or track mode switches
- Automatically modify internal flags or buffers along with training/eval mode

---

## How It Works

- `train(mode: bool = True)` sets `self.training = mode` for the module
- `eval()` is equivalent to `train(False)`
- Default implementation recursively calls `train(mode)` on all submodules
- Overriding allows custom behavior while keeping recursive updates intact


In [11]:
import torch
import torch.nn as nn
from torch import Tensor
from typing import Self

class CustomBlock(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.linear = nn.Linear(4, 4)
        self.dropout = nn.Dropout(p=0.5)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear(x)
        x = torch.relu(x)
        x = self.dropout(x)
        return x

    # -----------------------------
    # Override train() method
    # -----------------------------
    def train(self, mode: bool = True) -> Self:
        print(f"CustomBlock set to {'train' if mode else 'eval'} mode")
        super().train(mode)  # Call original method to update submodules
        # Add any custom logic here
        return self

    # -----------------------------
    # Override eval() method
    # -----------------------------
    def eval(self) -> Self:
        print("CustomBlock set to eval mode")
        return super().eval()


# Example usage
model = CustomBlock()
x = torch.randn(2, 4)

# Switch to training mode
model.train()
output_train = model(x)

# Switch to evaluation mode
model.eval()
with torch.no_grad():
    output_eval = model(x)

print("Output training mode:", output_train)
print("Output eval mode:", output_eval)


CustomBlock set to train mode
CustomBlock set to eval mode
CustomBlock set to eval mode
Output training mode: tensor([[0.0000, 0.0000, 0.0000, 0.0310],
        [0.0000, 0.0000, 1.8042, 0.0000]], grad_fn=<MulBackward0>)
Output eval mode: tensor([[0.0000, 0.0000, 0.3878, 0.0155],
        [0.0000, 0.0000, 0.9021, 1.4065]])


# Common Module Aggregators in PyTorch

When building neural networks, it is often useful to group multiple layers or submodules together.  
PyTorch provides several **module aggregators** that help organize layers and blocks. The most common ones are:



## `nn.Sequential`

- Holds modules in a sequential order.
- Executes them **in the order they are added** during the forward pass.
- Ideal for simple **stacked layers** with a single input and output.

**Key points:**

- Forward pass is automatically defined.
- Cannot handle multiple inputs or branching.

In [12]:
seq_model = nn.Sequential(
    nn.Linear(4, 8),
    nn.ReLU(),
    nn.Linear(8, 2)
)

x = torch.randn(5, 4)
output_seq = seq_model(x)
print("nn.Sequential output shape:", output_seq.shape)


nn.Sequential output shape: torch.Size([5, 2])


## `nn.ModuleList`

- Holds a **list of modules**.
- Does **not define a forward pass automatically**.
- Useful when you need to **loop over modules**, or have conditional computation.

**Key points:**

- Modules are registered properly, so parameters are tracked.
- You must define your own `forward()`.

In [13]:
class ModuleListModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(4, 8),
            nn.ReLU(),
            nn.Linear(8, 2)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

ml_model = ModuleListModel()
output_ml = ml_model(x)
print("nn.ModuleList output shape:", output_ml.shape)

nn.ModuleList output shape: torch.Size([5, 2])


## `nn.ModuleDict`

- Holds modules in a **dictionary** with string keys.
- Useful for architectures with **named branches**, **dynamic selection**, or **multi-head outputs**.
- Like `ModuleList`, it does **not define a forward pass**.

In [14]:
class ModuleDictModel(nn.Module):
    def __init__(self) -> None:
        super().__init__()
        self.branches = nn.ModuleDict({
            "branch1": nn.Linear(4, 8),
            "branch2": nn.Linear(4, 8)
        })
        self.output: nn.Linear = nn.Linear(8, 2)

    def forward(self, x: torch.Tensor, branch_name: str = "branch1") -> torch.Tensor:
        x = self.branches[branch_name](x)
        return self.output(x)

md_model = ModuleDictModel()
output_md = md_model(x, branch_name="branch2")
print("nn.ModuleDict output shape:", output_md.shape)

nn.ModuleDict output shape: torch.Size([5, 2])


## Homework

2 задания:
1. Реализуйте требуемый в заголовке блок (максмсум 0.8 балов).

## ResNet Block (0.1 балл)

![Resnet](assets/ResBlock.png)

https://arxiv.org/pdf/1512.03385

In [17]:
import torch.nn.functional as F

In [20]:
class ResidualBlock(nn.Module):
    """
    ResNet residual block: output = x + F(x),
    where F(x) = layer2(layer1(x)) and identity path is x.
    """

    def __init__(self, in_channels: int, out_channels: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(
            in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False
        )
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(
            out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False
        )
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        identity = self.shortcut(x)
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + identity
        out = F.relu(out)
        return out

In [21]:
block = ResidualBlock(in_channels=64, out_channels=64)
x = torch.randn(2, 64, 32, 32)
y = block(x)
print(y.shape)  # torch.Size([2, 64, 32, 32])

torch.Size([2, 64, 32, 32])


## Depthwise Separable Convolution (0.1 балл)
![DepthWiseConv](assets/DepthWiseConv.png)

https://arxiv.org/pdf/1610.02357

In [23]:
class SeparableConv2d(nn.Module):
    """
    Depthwise Separable Convolution:
    1) Depthwise: k×k conv per channel (M channels ==> M maps), then
    2) Pointwise: 1×1 conv across channels (M ==> N).
    """

    def __init__(
        self,
        in_channels: int,
        out_channels: int,
        kernel_size: int,
        stride: int = 1,
        padding: int | None = None,
        bias: bool = False,
    ):
        super().__init__()
        if padding is None:
            padding = (kernel_size - 1) // 2

        self.depthwise = nn.Conv2d(
            in_channels,
            in_channels,
            kernel_size=kernel_size,
            stride=stride,
            padding=padding,
            groups=in_channels,
            bias=bias,
        )
        
        self.pointwise = nn.Conv2d(
            in_channels,
            out_channels,
            kernel_size=1,
            stride=1,
            padding=0,
            bias=bias,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.depthwise(x)
        x = self.pointwise(x)
        return x

In [24]:
layer = SeparableConv2d(in_channels=32, out_channels=64, kernel_size=3, stride=1)
x = torch.randn(2, 32, 28, 28)
y = layer(x)
print(y.shape)  # torch.Size([2, 64, 28, 28])

torch.Size([2, 64, 28, 28])


## Vanilla Attention (0.1 балл)

Let:

$$
\text{query} \in \mathbb{R}^{B \times d} \\
\text{key} \in \mathbb{R}^{B \times L \times d}
$$

---

### Alignment Scores

$$
\text{score} = \text{key} \cdot (W_\text{align} \, \text{query})^T \\
\text{score} \in \mathbb{R}^{B \times L}
$$

---

### Attention Weights

$$
\text{att} = \text{softmax}(\text{score}, \text{dim}=1) \\
\text{att} \in \mathbb{R}^{B \times L}
$$

---

### Context Vector

$$
\text{context} = \sum_{i=1}^{L} \text{att}_i \cdot \text{key}_i \\
\text{context} \in \mathbb{R}^{B \times d}
$$

---

### Output

$$
\text{out} = \tanh(W_\text{value} \, \text{context} + W_\text{query} \, \text{query}) \\
\text{out} \in \mathbb{R}^{B \times d}
$$



https://arxiv.org/abs/1409.0473


https://arxiv.org/abs/1508.04025

In [25]:
class VanillaAttention(nn.Module):
    """
    Vanilla (additive) attention: query (B, d), key (B, L, d).
    score = key @ (W_align @ query)^T, att = softmax(score), context = att @ key,
    out = tanh(W_value @ context + W_query @ query).
    """

    def __init__(self, d: int):
        super().__init__()
        self.d = d
        self.W_align = nn.Linear(d, d)
        self.W_value = nn.Linear(d, d)
        self.W_query = nn.Linear(d, d)

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
    ) -> torch.Tensor:
        
        query_proj = self.W_align(query)  # (B, d)
        score = torch.bmm(key, query_proj.unsqueeze(-1)).squeeze(-1)  # (B, L)
        att = F.softmax(score, dim=1)  # (B, L)
        context = torch.bmm(att.unsqueeze(1), key).squeeze(1)  # (B, d)
        out = torch.tanh(self.W_value(context) + self.W_query(query))  # (B, d)
        return out

In [26]:
B, L, d = 4, 10, 8
attn = VanillaAttention(d)
query = torch.randn(B, d)
key = torch.randn(B, L, d)
out = attn(query, key)
print(out.shape)  # torch.Size([4, 8])

torch.Size([4, 8])


## Dot Product Attention (0.1 балл)

$$
Q \in \mathbb{R}^{B \times L_q \times d_k} \\
K \in \mathbb{R}^{B \times L_k \times d_k} \\
V \in \mathbb{R}^{B \times L_k \times d_k}
$$

$$
S = \frac{Q K^T}{\sqrt{d_k}}
$$

$$
\text{Attention}(Q, K, V) = \text{softmax}(S, \text{dim}=-1) \, V
$$



https://arxiv.org/abs/1706.03762


In [27]:
class DotProductAttention(nn.Module):
    """
    Scaled dot-product attention: S = Q K^T / sqrt(d_k), out = softmax(S, dim=-1) @ V.
    """

    def __init__(self, d_k: int):
        super().__init__()
        self.d_k = d_k
        self.scale = d_k ** (-0.5)

    def forward(
        self,
        Q: torch.Tensor,
        K: torch.Tensor,
        V: torch.Tensor,
    ) -> torch.Tensor:
        
        S = torch.bmm(Q, K.transpose(-2, -1)) * self.scale  # (B, L_q, L_k)
        att = F.softmax(S, dim=-1)  # (B, L_q, L_k)
        out = torch.bmm(att, V)  # (B, L_q, d_k)
        return out

In [28]:
B, L_q, L_k, d_k = 2, 5, 7, 8
attn = DotProductAttention(d_k)
Q = torch.randn(B, L_q, d_k)
K = torch.randn(B, L_k, d_k)
V = torch.randn(B, L_k, d_k)
out = attn(Q, K, V)
print(out.shape)  # torch.Size([2, 5, 8])

torch.Size([2, 5, 8])


## Multihead Attention (0.1 балл)

![MultiheadAttention](assets/MultiheadAttention.webp)

https://arxiv.org/abs/1706.03762


In [34]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Scaled Dot-Product Attention.
    h heads, d_model dimension; per-head dim d_k = d_model / h.
    """

    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.scale = self.d_k ** (-0.5)

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        Q: torch.Tensor,
        K: torch.Tensor,
        V: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        # Q, K, V: (B, L, d_model)
        B, L_q, _ = Q.shape
        _, L_k, _ = K.shape

        Q = self.W_q(Q).view(B, L_q, self.num_heads, self.d_k).transpose(1, 2)   # (B, h, L_q, d_k)
        K = self.W_k(K).view(B, L_k, self.num_heads, self.d_k).transpose(1, 2)   # (B, h, L_k, d_k)
        V = self.W_v(V).view(B, L_k, self.num_heads, self.d_k).transpose(1, 2)   # (B, h, L_k, d_k)

        S = torch.matmul(Q, K.transpose(-2, -1)) * self.scale   # (B, h, L_q, L_k)
        if mask is not None:
            S = S.masked_fill(mask == 0, float("-inf"))
        att = F.softmax(S, dim=-1)
        att = self.dropout(att)

        out = torch.matmul(att, V)   # (B, h, L_q, d_k)
        out = out.transpose(1, 2).contiguous().view(B, L_q, self.d_model)
        return self.W_o(out)

In [35]:

B, L, d_model, h = 2, 10, 64, 4
mha = MultiHeadAttention(d_model=d_model, num_heads=h)
x = torch.randn(B, L, d_model)
out = mha(x, x, x)
print("MultiHeadAttention out shape:", out.shape)   # (2, 10, 64)

MultiHeadAttention out shape: torch.Size([2, 10, 64])


## Transformer Encoder Layer (0.1 балл)


![Transformer Encoder Layer](assets/TransformerEncoder.png)


https://arxiv.org/abs/1706.03762

In [30]:
class TransformerEncoderLayer(nn.Module):
    """
    Transformer Encoder: Multi-Head Self-Attention + residual + LayerNorm,
    then FFN (two linear + activation) + residual + LayerNorm.
    """

    def __init__(
        self,
        d_model: int,
        num_heads: int,
        d_ff: int | None = None,
        dropout: float = 0.0,
        activation: str = "relu",
    ):
        super().__init__()
        d_ff = d_ff or 4 * d_model

        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU() if activation == "relu" else nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        x: torch.Tensor,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        # x: (B, L, d_model)
        x = x + self.dropout(self.self_attn(x, x, x, mask))
        x = self.norm1(x)
        x = x + self.ffn(x)
        x = self.norm2(x)
        return x

In [36]:
enc = TransformerEncoderLayer(d_model=64, num_heads=4, d_ff=128)
x = torch.randn(2, 10, 64)
out = enc(x)
print("TransformerEncoderLayer out shape:", out.shape)   # (2, 10, 64)

TransformerEncoderLayer out shape: torch.Size([2, 10, 64])


## MLP Mixer (0.1 балл)


![MLPMixer](assets/MLPMixer.png)


https://arxiv.org/abs/2105.01601

In [37]:
class MLPMixerBlock(nn.Module):
    """
    One Mixer layer: token-mixing MLP (over S) + residual, then channel-mixing MLP (over C) + residual.
    """

    def __init__(
        self,
        num_patches: int,
        channels: int,
        token_hidden_dim: int,
        channel_hidden_dim: int,
        dropout: float = 0.0,
    ):
        super().__init__()
        self.norm1 = nn.LayerNorm(channels)
        self.token_mix = nn.Sequential(
            nn.Linear(num_patches, token_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(token_hidden_dim, num_patches),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(channels)
        self.channel_mix = nn.Sequential(
            nn.Linear(channels, channel_hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(channel_hidden_dim, channels),
            nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, S, C)
        y = self.norm1(x)
        y = x.transpose(1, 2)           # (B, C, S) — token_mix по последней оси S
        y = self.token_mix(y)           # (B, C, S)
        y = y.transpose(1, 2)           # (B, S, C)
        x = x + y

        x = x + self.channel_mix(self.norm2(x))   # channel_mix по C
        return x

In [38]:

B, S, C = 2, 16, 64
mixer = MLPMixerBlock(num_patches=S, channels=C, token_hidden_dim=32, channel_hidden_dim=128)
x = torch.randn(B, S, C)
out = mixer(x)
print("MLPMixerBlock out shape:", out.shape)   # (2, 16, 64)

MLPMixerBlock out shape: torch.Size([2, 16, 64])


## ConvMixer (0.1 балл)

![ConvMixer](assets/ConvMixer.png)


https://arxiv.org/abs/2201.09792

In [39]:
class ConvMixerBlock(nn.Module):
    """One block: (Depthwise Conv -> ReLU -> BN) + residual, then (Pointwise Conv -> ReLU -> BN) + residual."""

    def __init__(self, dim: int, kernel_size: int):
        super().__init__()
        self.dw = nn.Sequential(
            nn.Conv2d(dim, dim, kernel_size, padding=kernel_size // 2, groups=dim),
            nn.GELU(),
            nn.BatchNorm2d(dim),
        )
        self.pw = nn.Sequential(
            nn.Conv2d(dim, dim, 1),
            nn.GELU(),
            nn.BatchNorm2d(dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.dw(x)
        x = x + self.pw(x)
        return x


class ConvMixer(nn.Module):
    """
    ConvMixer: patch embedding + depth x (Depthwise residual + Pointwise residual).
    """

    def __init__(
        self,
        dim: int,
        depth: int,
        patch_size: int = 7,
        kernel_size: int = 9,
        num_classes: int = 1000,
    ):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, dim, kernel_size=patch_size, stride=patch_size),
            nn.GELU(),
            nn.BatchNorm2d(dim),
        )
        self.blocks = nn.Sequential(*[ConvMixerBlock(dim, kernel_size) for _ in range(depth)])
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        return x

In [40]:
model = ConvMixer(dim=64, depth=4, patch_size=2, kernel_size=5, num_classes=10)
x = torch.randn(2, 3, 32, 32)
out = model(x)
print("ConvMixer out shape:", out.shape)   # (2, 10)

ConvMixer out shape: torch.Size([2, 10])


## Вопрос (0.2 балла)

Объясните, почему MLPMixer, ConvMixer может работать почти так же эффективно, как обычный Multihead Attention.

Напишите формулу, связывающую Multihead Attention, ConvMixer и MLPMixer

Опишите преимущества и недостатки между ConvMixer, MLPMixer и Multihead Attention

---

**Ответ:**

Все три подхода делают одно и то же по смыслу: **смешивание по двум осям** — по позициям (токены/патчи) и по каналам (признакам). Отличие только в том, *чем* смешивать: в Attention — взвешенное усреднение (content-dependent), в MLPMixer — два MLP (по токенам и по каналам), в ConvMixer — depthwise (позиции) и pointwise (каналы). Поэтому MLPMixer и ConvMixer могут работать почти так же эффективно при меньшей стоимости.

**Формула:** общая схема одного «слоя»:
$$\text{Output} = \text{MixChannels}\bigl(\text{MixPositions}(X) + X\bigr) + \ldots$$
- **Attention:** MixPositions = softmax(QK^T/√d)V, MixChannels = линейные проекции Q,K,V и выхода.
- **MLPMixer:** MixPositions = MLP по оси патчей, MixChannels = MLP по оси каналов.
- **ConvMixer:** MixPositions = Depthwise Conv, MixChannels = Pointwise Conv (1×1).

**Плюсы и минусы:**
- **Multihead Attention:** плюсы — гибкое, content-dependent смешивание, SOTA; минусы — O(L²) по длине, дорого по памяти и вычислениям.
- **MLPMixer:** плюсы — проще и быстрее Transformer, фиксированный MLP; минусы — нет адаптивных весов как в attention.
- **ConvMixer:** плюсы — индуктивный bias свёрток (локальность), мало параметров, быстрый; минусы — слабее глобальный контекст без больших ядер.